In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("login", "")
login = dbutils.widgets.get("login")

In [0]:
catalog = "dbr_dev"
schema = f"{login}_bronze"
table = "stock_prices"
volume_path = f"/Volumes/{catalog}/{schema}/landing"
table_name = f"{catalog}.{schema}.{table}"

In [0]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{volume_path}/*.csv")
    # .select("_metadata")
)
# df_raw.printSchema()
# df_raw.show(3, truncate=False)

In [0]:
df_bronze = (
    df_raw
    .withColumnRenamed("Adj Close", "Adj_close")
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("symbol", F.upper(F.regexp_replace(F.col("_metadata.file_name"), r"\.csv$", "")))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

In [0]:
if not spark.catalog.tableExists(table_name):
    df_bronze.write.format("delta").saveAsTable(table_name)

    print(f"Table {table_name} created.")
else:
    delta_table = DeltaTable.forName(spark, table_name)
    (delta_table.alias("t").merge(
        df_bronze.alias("s"),
        "t.symbol = s.symbol AND t.Date = s.Date")
    .whenNotMatchedInsertAll()
    .execute()
    )
    print(f"Table {table_name} merged.")

In [0]:
display(spark.sql(f"""
                  SELECT symbol, COUNT(*) AS count, MIN(Date) as from, max(Date) as to
                  FROM {table_name}
                  GROUP BY symbol
                  ORDER BY symbol DESC
                  """))

In [0]:
display(spark.sql(f"SELECT symbol, Date, Close, source_file, ingestion_timestamp, load_date FROM {table_name} LIMIT 5"))